# Only the missing cells

These eight code cells add data preparation and training to your two component notebooks. No encoder, anchors, classifier, fusion, LACR, or loss definitions are copied here. Imports already present in those notebooks are omitted.

**Insert them in the same notebook/kernel:**

- Run your original encoder notebook through its imports, labels, helper functions, and configuration first. This provides `torch`, `np`, `pd`, `dataclass`, and `device`.
- Insert cells **1–2** below before the original adjacency cell, which needs `train_labels`.
- Finish your original encoder/anchors cells, then run your original classifier/fusion/loss cells.
- Append cells **3–8** below after the classifier notebook's forward-pass check.

Keep `held_out_classes=[]` in both originals. Cell 2 supplies `train`, `train_labels`, and `class_counts`; if you already prepare those elsewhere, use one consistent data split instead of preparing them twice. This version uses the fixed-anchor model in the two component notebooks.


In [ ]:
# 1. Additional imports and training settings
import json
import math
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
from contextlib import nullcontext

from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

@dataclass
class TrainingSettings:
    epochs: int = 7
    batch_size: int = 8
    eval_batch_size: int = 32
    grad_accum_steps: int = 4
    encoder_lr: float = 2e-5
    head_lr: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    patience: int = 3
    min_delta: float = 0.0
    max_len: int = 64
    seed: int = 42
    use_amp: bool = True
    bounded_loss: bool = False

training = TrainingSettings()
assert training.batch_size > 0 and training.grad_accum_steps > 0
assert training.epochs > 0 and training.eval_batch_size > 0 and training.patience > 0
assert 0 <= training.warmup_ratio < 1 and training.min_delta >= 0


In [ ]:
# 2. Data needed before the original adjacency and model-check cells
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)

def multi_hot(label_lists):
    targets = np.zeros((len(label_lists), num_labels), dtype=np.float32)
    for result_row, labels in enumerate(label_lists):
        targets[result_row, labels] = 1.0
    return targets

# These names are required by your original notebooks.
train = dataset["train"].to_pandas()
train_labels = multi_hot(dataset["train"]["labels"])
class_counts = train_labels.sum(axis=0)
assert (class_counts > 0).all()
print({split: len(dataset[split]) for split in dataset})


In [ ]:
# 3. Data loaders — after your original model is built
required = ["model", "tokenizer", "cfg", "GOEMOTIONS_LABELS", "build_loss_fn"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Run the original model cells in this same notebook first. Missing: {missing}")
assert label_names == GOEMOTIONS_LABELS, "Dataset and model label order must match."
assert not cfg.held_out_classes, "This training setup supervises all labels."
assert torch.all(model.fusion.seen_mask == 1), "Rebuild the model without held-out labels."
assert training.seed == cfg.seed, "Use the same seed in training and original model settings."

use_mixed_precision = training.use_amp and device.type == "cuda"
output_dir = Path("goemotions_runs") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output_dir.mkdir(parents=True, exist_ok=False)
print("Device:", device)
print("Effective batch size:", training.batch_size * training.grad_accum_steps)
print("Results folder:", output_dir.resolve())

def prepare_batch(batch):
    encoded = tokenizer(batch["text"], truncation=True, max_length=training.max_len)
    encoded["labels"] = multi_hot(batch["labels"]).tolist()
    return encoded

tokenized = {
    split: dataset[split].map(prepare_batch, batched=True,
                              remove_columns=dataset[split].column_names)
    for split in ("train", "validation", "test")
}

def collate_batch(examples):
    features = [{k: example[k] for k in ("input_ids", "attention_mask")}
                for example in examples]
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor([example["labels"] for example in examples],
                                   dtype=torch.float32)
    return batch

train_loader = DataLoader(tokenized["train"], batch_size=training.batch_size,
    shuffle=True, generator=torch.Generator().manual_seed(training.seed),
    collate_fn=collate_batch, pin_memory=device.type == "cuda")
val_loader = DataLoader(tokenized["validation"], batch_size=training.eval_batch_size,
    shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(tokenized["test"], batch_size=training.eval_batch_size,
    shuffle=False, collate_fn=collate_batch)


In [ ]:
# 4. Create the loss instance and optimizer
# Use the loss already defined in your classifier notebook.
seen_mask = model.fusion.seen_mask.detach().cpu()
if training.bounded_loss:
    loss_fn = BoundedMaskedLDAMASLLoss(
        class_counts, seen_mask, loss_type=cfg.loss_type,
        ldam_max_margin=cfg.ldam_max_margin, gamma_pos=cfg.gamma_pos,
        gamma_neg=cfg.gamma_neg, prob_margin=cfg.prob_margin,
        reduction=cfg.loss_reduction, logit_clip=cfg.logit_clip,
        logit_penalty=cfg.logit_penalty).to(device)
else:
    loss_fn = build_loss_fn(cfg, class_counts, seen_mask).to(device)

print("Loss:", cfg.loss_type, "| bounded:", training.bounded_loss)
tokenizer.save_pretrained(output_dir / "tokenizer")
(output_dir / "settings.json").write_text(json.dumps({
    "training": asdict(training), "classifier": asdict(cfg),
    "encoder_config": model.encoder.backbone.config.to_dict(),
    "label_names": label_names, "class_counts": class_counts.tolist(),
    "dataset": "google-research-datasets/go_emotions", "dataset_config": "simplified",
    "split_fingerprints": {s: dataset[s]._fingerprint for s in dataset},
    "anchor_policy": "use the existing fixed anchor bank without refreshing",
    "probabilities": "sigmoid(raw z_final)"
}, indent=2))

# Fine-tune RoBERTa slowly and let the new layers learn faster.
# Skip weight decay for biases and normalization parameters.
parameter_groups = {}
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    learning_rate = training.encoder_lr if name.startswith("encoder.") else training.head_lr
    weight_decay = 0.0 if parameter.ndim <= 1 or name.endswith("bias") else training.weight_decay
    parameter_groups.setdefault((learning_rate, weight_decay), []).append(parameter)
optimizer = torch.optim.AdamW([
    {"params": parameters, "lr": learning_rate, "weight_decay": weight_decay}
    for (learning_rate, weight_decay), parameters in parameter_groups.items()])
updates_per_epoch = math.ceil(len(train_loader) / training.grad_accum_steps)
total_updates = updates_per_epoch * training.epochs
warmup_updates = int(total_updates * training.warmup_ratio)

def learning_rate_schedule(step):
    if step < warmup_updates:
        return float(step + 1) / max(1, warmup_updates)
    return max(0.0, (total_updates - step) / max(1, total_updates - warmup_updates))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, learning_rate_schedule)
scaler = torch.amp.GradScaler("cuda", enabled=use_mixed_precision)


In [ ]:
# 5. Training and evaluation functions
def autocast_context():
    return torch.autocast("cuda", dtype=torch.float16) if use_mixed_precision else nullcontext()

def to_device(batch):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}

def calculate_metrics(targets, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(np.int32)
    result = {"subset_accuracy": float(accuracy_score(targets, predictions))}
    for average in ("micro", "macro"):
        precision, recall, f1, _ = precision_recall_fscore_support(
            targets, predictions, average=average, zero_division=0)
        result.update({f"precision_{average}": float(precision),
                       f"recall_{average}": float(recall), f"f1_{average}": float(f1)})
    return result

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    # Turn off dropout while checking predictions.
    model.eval()
    total_loss, total_examples = 0.0, 0
    all_probabilities, all_targets = [], []
    for batch in loader:
        batch = to_device(batch)
        with autocast_context():
            logits = model(batch["input_ids"], batch["attention_mask"])["z_final"]
        loss = loss_fn(logits.float(), batch["labels"])
        if not torch.isfinite(loss) or not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite evaluation loss/logits.")
        batch_size = len(batch["labels"])
        total_loss += loss.item() * batch_size
        total_examples += batch_size
        all_probabilities.append(logits.float().sigmoid().cpu().numpy())
        all_targets.append(batch["labels"].cpu().numpy())
    if total_examples == 0:
        raise ValueError("Evaluation loader is empty.")
    return total_loss / total_examples, np.concatenate(all_probabilities), np.concatenate(all_targets)

def train_one_epoch(model, loader, optimizer, scheduler, scaler, loss_fn, epoch):
    # Turn on training behavior, including dropout.
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss, total_examples, examples_since_update, skipped_updates = 0.0, 0, 0, 0
    progress = tqdm(loader, desc=f"Epoch {epoch}/{training.epochs}")
    for batch_index, batch in enumerate(progress):
        batch = to_device(batch)
        batch_size = len(batch["labels"])
        with autocast_context():
            logits = model(batch["input_ids"], batch["attention_mask"])["z_final"]
        # Calculate loss in float32 to avoid numerical issues.
        loss = loss_fn(logits.float(), batch["labels"])
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss at batch {batch_index}.")
        # Add gradients across batches before updating the weights.
        # Counting examples also handles the smaller batch at the end.
        scaler.scale(loss * batch_size).backward()
        examples_since_update += batch_size
        total_loss += loss.item() * batch_size
        total_examples += batch_size
        ready_to_update = (batch_index + 1) % training.grad_accum_steps == 0 or batch_index + 1 == len(loader)
        if ready_to_update:
            # Average the gradients, clip them, then update the model.
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.div_(examples_since_update)
            torch.nn.utils.clip_grad_norm_(model.parameters(), training.max_grad_norm,
                                            error_if_nonfinite=not use_mixed_precision)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()  # Only advance after an actual weight update.
            else:
                skipped_updates += 1
            optimizer.zero_grad(set_to_none=True)
            examples_since_update = 0
        progress.set_postfix(loss=f"{total_loss / total_examples:.4f}")
    return total_loss / total_examples, skipped_updates


In [ ]:
# 6. Run training
best_score, best_epoch, epochs_without_improvement = -1.0, 0, 0
history = []
checkpoint_path = output_dir / "best_model.pt"
for epoch in range(1, training.epochs + 1):
    train_loss, skipped_updates = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, loss_fn, epoch)
    val_loss, val_probabilities, val_targets = evaluate(model, val_loader, loss_fn)
    val_metrics = calculate_metrics(val_targets, val_probabilities, 0.5)
    result_row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
           "skipped_amp_updates": skipped_updates,
           **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(result_row)
    pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
    # Keep the model that performs best on validation.
    score = val_metrics["f1_macro"]
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, val loss={val_loss:.4f}, "
          f"val Macro-F1={score:.4f}, Micro-F1={val_metrics['f1_micro']:.4f}")
    if score > best_score + training.min_delta:
        best_score, best_epoch, epochs_without_improvement = score, epoch, 0
        # Save CPU weights so they can also be loaded without a GPU.
        # Finish writing before replacing the previous checkpoint.
        saved_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        temporary_path = output_dir / "best_model.tmp"
        torch.save(saved_weights, temporary_path)
        temporary_path.replace(checkpoint_path)
        del saved_weights
        (output_dir / "best_validation.json").write_text(json.dumps(
            {"epoch": best_epoch, "threshold": 0.5, **val_metrics}, indent=2))
        print("Saved new best model.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= training.patience:
            print(f"Early stopping: {training.patience} epochs without improvement.")
            break
print(f"Best epoch: {best_epoch}; validation Macro-F1 @ 0.5: {best_score:.4f}")


In [ ]:
# 7. Select a threshold using validation
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
val_loss, val_probabilities, val_targets = evaluate(model, val_loader, loss_fn)
threshold_results = []
for threshold in np.linspace(0.05, 0.95, 19):
    threshold_results.append({"threshold": float(threshold),
        **calculate_metrics(val_targets, val_probabilities, float(threshold))})
# If two thresholds tie, choose the one closer to 0.5.
best_result = max(threshold_results,
    key=lambda recall: (recall["f1_macro"], -abs(recall["threshold"] - 0.5)))
best_threshold = best_result["threshold"]
pd.DataFrame(threshold_results).to_csv(output_dir / "validation_thresholds.csv", index=False)
(output_dir / "inference_config.json").write_text(json.dumps({
    "threshold": best_threshold, "label_names": label_names,
    "best_epoch": best_epoch, "validation_metrics": best_result,
    "probabilities": "sigmoid(raw z_final)", "comparison": ">="}, indent=2))
print("Validation-selected threshold:", best_threshold)
pd.DataFrame(threshold_results)


In [ ]:
# 8. Final test evaluation
test_loss, test_probabilities, test_targets = evaluate(model, test_loader, loss_fn)
test_metrics = calculate_metrics(test_targets, test_probabilities, best_threshold)
report = {"best_epoch": best_epoch, "threshold": best_threshold,
          "test_loss": test_loss, **test_metrics}
(output_dir / "test_metrics.json").write_text(json.dumps(report, indent=2))
test_predictions = (test_probabilities >= best_threshold).astype(np.int32)
precision, recall, f1, support = precision_recall_fscore_support(
    test_targets, test_predictions, average=None, zero_division=0)
per_label = pd.DataFrame({"label": label_names, "precision": precision, "recall": recall,
                         "f1": f1, "support": support.astype(int)})
per_label.to_csv(output_dir / "test_per_label.csv", index=False)
print(json.dumps(report, indent=2))
print("Files saved in:", output_dir.resolve())
per_label
